# P&ID Stage 2 — FGC / ArcFace train on Colab (GPU)

Fine-grained classification: phân biệt các symbol TRONG cùng một coarse group (16 loại
valve, 3 loại blind_disc, ...) bằng **ResNet-34 + ArcFace** (metric learning + 'Other' gate).

**Độc lập Stage 1**: FGC train trên GT crop (cắt từ ground-truth, không dùng detection của
detector). Chỉ cần `data/canonical/` trên Drive (đã upload cho Stage 1).

**Chuẩn bị**: code đã push GitHub (nhánh có ArcFace), `data/canonical/` đã ở Drive.
**Runtime**: GPU.

## 1. GPU + mount Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/PID_Recognition'   # <-- sửa nếu path khác
DATA_ROOT  = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
assert os.path.isdir(f'{DATA_ROOT}/canonical'), f'Không thấy {DATA_ROOT}/canonical'
print('canonical images:', len(os.listdir(f'{DATA_ROOT}/canonical/images')))

## 2. Clone code + cài deps

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'   # <-- sửa
BRANCH   = 'feat/prepare-for-colab-stage-1'                       # <-- nhánh có ArcFace
%cd /content
![ -d pid ] || git clone -b $BRANCH $REPO_URL pid
%cd /content/pid
!git checkout $BRANCH && git pull
!pip install -q uv
!uv pip install --system -q timm torch torchvision hydra-core omegaconf pycocotools \
    opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml
os.environ['PYTHONPATH'] = '/content/pid/src'

## 3. Prep crops — cắt GT crop theo fgc_group (chạy 1 LẦN)

`extract_crops` đọc canonical images + COCO, cắt một PNG 128px cho mỗi GT symbol thuộc
4 fgc_group (valve / blind_disc / heat_exchanger / instrument) vào `derived/crops_fgc/`.
Cần classes.yaml + manifest (Stage 1 đã sinh; nếu chưa, chạy prep classes/manifest trước).

> Output lưu Drive -> session sau skip. Cắt từ ảnh 7168px nên hơi chậm (I/O) nhưng crop nhỏ.

In [ ]:
import os
crops_marker = f'{DATA_ROOT}/derived/crops_fgc/train'
if os.path.isdir(crops_marker) and os.listdir(crops_marker):
    print('Crops đã có trên Drive, skip:', crops_marker)
else:
    # đảm bảo classes.yaml + manifest tồn tại (idempotent nếu đã có)
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step classes data_root=$DATA_ROOT
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step manifest data_root=$DATA_ROOT
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step crops data_root=$DATA_ROOT
    print('Crops xong.')

## 3b. Copy crops Drive → local (tránh I/O chậm khi train)

Giống Stage 1: train đọc crop từ local `/content` nhanh hơn Drive nhiều. Crops nhỏ (128px)
nên zip/copy nhanh. Train dùng `data_root=$LOCAL_DATA_ROOT`.

In [ ]:
import shutil, glob, time, os
LOCAL_DATA_ROOT = '/content/data'
LOCAL_CROPS = f'{LOCAL_DATA_ROOT}/derived/crops_fgc'
DRIVE_CROPS = f'{DATA_ROOT}/derived/crops_fgc'
DRIVE_ZIP   = f'{DATA_ROOT}/derived/crops_fgc.zip'
t0 = time.time()
os.makedirs(f'{LOCAL_DATA_ROOT}/derived', exist_ok=True)
# classes.yaml cũng cần ở local data_root (build_dataset đọc qua DataPaths)
os.makedirs(f'{LOCAL_DATA_ROOT}/canonical', exist_ok=True)
shutil.copy2(f'{DATA_ROOT}/canonical/classes.yaml', f'{LOCAL_DATA_ROOT}/canonical/classes.yaml')
if os.path.exists(DRIVE_ZIP):
    !unzip -q -o "$DRIVE_ZIP" -d "$LOCAL_DATA_ROOT/derived/"
elif not os.path.isdir(LOCAL_CROPS):
    shutil.copytree(DRIVE_CROPS, LOCAL_CROPS)
print(f'Crops local sau {time.time()-t0:.0f}s')
for split in ('train','val'):
    n = len(glob.glob(f'{LOCAL_CROPS}/{split}/*/*.png'))
    print(f'  {split}: {n} crops,', len(glob.glob(f'{LOCAL_CROPS}/{split}/*')), 'fine classes')

In [ ]:
# (TÙY CHỌN, 1 lần) zip crops lên Drive để session sau giải nén nhanh
import os
if not os.path.exists(DRIVE_ZIP):
    !cd "$LOCAL_DATA_ROOT/derived" && zip -q -r crops_fgc.zip crops_fgc && cp crops_fgc.zip "$DRIVE_ZIP"
    print('Đã tạo', DRIVE_ZIP)
else:
    print('crops_fgc.zip đã có')

## 4. Train FGC (ArcFace) — một head per fgc_group

`metric_learning=true` (mặc định) -> ArcFace. Train 4 head riêng (valve/blind_disc/
heat_exchanger/instrument), mỗi head lưu `<group>.pt` + class_centers vào
`runs/<ts>/fgc_checkpoints/`. Checkpoint ra Drive (bền). FGC nhẹ (crop 128px) -> nhanh.

In [ ]:
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR \
    fgc.metric_learning=true fgc.epochs=60 fgc.batch=128

## 4b. Resume FGC (nếu Colab ngắt)

Sửa `FGC_RESUME` = thư mục fgc_checkpoints phiên trước. Mỗi group nạp model+optimizer+epoch,
train tiếp; group đã xong bỏ qua.

In [ ]:
FGC_RESUME = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/fgc_checkpoints'   # <-- sửa
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR \
    fgc.metric_learning=true fgc.epochs=60 fgc.resume_from=$FGC_RESUME

## 5. Eval FGC (trên GT crop, độc lập detector)

Đo within-group accuracy, top-k, rare-macro-F1, confusion matrix per group, + other_gate
false-reject-rate (ArcFace). Sửa `FGC_CKPT` = fgc_checkpoints vừa train.

In [ ]:
FGC_CKPT = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/fgc_checkpoints'   # <-- sửa
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR \
    eval.mode=det_plus_fgc eval.split=val \
    fgc.metric_learning=true fgc.ckpt_dir=$FGC_CKPT

# in metrics.json + confusion matrix per group
import json, glob
from IPython.display import Image as IPImage, display
run = sorted(glob.glob(f'{RUNS_DIR}/*/metrics.json'))[-1]
print('metrics:', run)
print(json.dumps(json.load(open(run)).get('fgc', {}), indent=2, ensure_ascii=False)[:2000])
for p in sorted(glob.glob(f'{os.path.dirname(run)}/confusion_*.png')):
    print(p.split('/')[-1]); display(IPImage(p))